Score commercial auto policies for expected claim risk
Fits a model on historical data and scores the current period

Data:
train.csv  - policies 2020-2022, includes claim outcomes
score.csv  - policies 2023-2024, no outcomes (to be scored)

In [159]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [160]:
# ---- load ----
df2 = pd.read_csv("./train.csv")
df3 = pd.read_csv("./score.csv")

print("train:", df2.shape)
print("score:", df3.shape)

train: (15075, 31)
score: (3000, 28)


In [161]:
# ---- preprocessing ----

# fill numeric missing with 0 -- good enough for now
df2[df2.select_dtypes("number").columns] = df2.select_dtypes("number")#.fillna(0) a missing risk_score_external becomes 0, which the data dictionary defines as "lowest possible risk,"
df3[df3.select_dtypes("number").columns] = df3.select_dtypes("number")#.fillna(0) same with prior_loss_amount, where a real 0 (no losses) becomes indistinguishable from "unknown."


# normalize business_type casing (Trucking, TRUCKING, trucking) it was inconsistent in the data
df2["business_type"] = df2["business_type"].str.strip().str.upper()
df3["business_type"] = df3["business_type"].str.strip().str.upper() 


print("Unique business types after normalization:")
print(sorted(df2["business_type"].unique()))

# encode categoricals for train
df2["coverage_type"] = df2["coverage_type"].astype("category").cat.codes
df2["business_type"] = df2["business_type"].astype("category").cat.codes
df2["state"] = df2["state"].astype("category").cat.codes

# same for score
df3["coverage_type"] = df3["coverage_type"].astype("category").cat.codes
df3["business_type"] = df3["business_type"].astype("category").cat.codes
df3["state"] = df3["state"].astype("category").cat.codes


Unique business types after normalization:
['AGRICULTURE', 'CONSTRUCTION', 'OTHER', 'RETAIL', 'SERVICE', 'TRUCKING']


In [162]:
claim_rate_by_type = (
    df2.assign(claim=(df2["claim_count"] > 0).astype(int))
       .groupby("business_type")["claim"]
       .agg(claim_rate="mean", n_policies="count", n_claims="sum")
       .sort_values("claim_rate", ascending=False)
)
print(claim_rate_by_type)

               claim_rate  n_policies  n_claims
business_type                                  
5                0.134039        4536       608
1                0.129408        3006       389
3                0.117398        3782       444
4                0.113189        2995       339
2                0.108289         748        81
0                0.000000           8         0


In [163]:
# ---- target ----
# using binary had-a-claim flag as target
df2["target"] = (df2["claim_count"] > 0).astype(int)
print("positive rate:", round(df2["target"].mean(), 3))

# 

positive rate: 0.123


In [164]:
# ---- features ----
feat_cols = [
    "coverage_type",
    "vehicle_count", "vehicle_avg_age", "driver_count", "driver_avg_age",
    "years_in_business", "prior_year_mileage_000",
    "business_type", "state",
    "prior_apd_claim_count", "prior_al_claim_count", "prior_loss_amount",
    "deductible", "coverage_limit_000", "annual_premium",
    "risk_score_external", "num_heavy_vehicles", "late_payment_count",
    #"claim_paid_amount_current_period", # data leakage -- this is created after the claim is paid. 
    # This cant exist at the moement you are trying to predicst the risk.
    #"days_to_first_claim_report", # deleting clos-info that is created after the claim is reported
]

X = df2[feat_cols]
y = df2["target"]

In [165]:
# ---- fit model ----
# random split is a wrong split in this case, because the data is time-based. 

df2["snapshot_date"] = pd.to_datetime(df2["snapshot_date"])
df2["snapshot_year"] = df2["snapshot_date"].dt.year

# fixed time split train on data from 2020 and 2021, test on data from 2022
train_mask = df2["snapshot_year"] <= 2021 # 2020,2021
test_mask = df2["snapshot_year"] == 2022 

X_train = X[train_mask]
y_train = y[train_mask]

X_test = X[test_mask]
y_test = y[test_mask]


from sklearn.impute import SimpleImputer

tmp = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),  # fill NaNs before scaling
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

tmp.fit(X, y)


from sklearn.ensemble import HistGradientBoostingClassifier

#tmp = HistGradientBoostingClassifier(
#max_iter=300,
#learning_rate=0.05,
#max_leaf_nodes=31,
#random_state=42
#)

#tmp.fit(X_train, y_train)

from sklearn.ensemble import RandomForestClassifier

#tmp = RandomForestClassifier(
#n_estimators=300,
#max_depth=8,
#min_samples_leaf=20,
#class_weight="balanced",
#random_state=42,
#n_jobs=-1
#)

#tmp.fit(X_train, y_train)

# the results are not better due to the overly simple dataset. 


In [166]:
# ---- evaluate ----
from sklearn.metrics import roc_auc_score, accuracy_score, average_precision_score


preds = tmp.predict(X_test)
probs = tmp.predict_proba(X_test)[:, 1]

print("accuracy:", accuracy_score(y_test, preds))
print("ROC-AUC:", roc_auc_score(y_test, probs))
print("PR-AUC:", average_precision_score(y_test, probs))

# The prototype fits on X_train, then calls .predict() and reports accuracy on X_train again — it never touches X_test. 
# and the next point is that the accuracy is a bad metrics here regardeless. only 12.4% of policies have a claim,
# # so a model that predicts "no claim" for every policy would have 87.6% accuracy. - its useless
# After removing leakage variables, the model performance dropped from perfect accuracy to a more realistic validation performance. 
# This confirmed that the prototype model was relying on post-claim information.

accuracy: 0.8736483780536644
ROC-AUC: 0.6840593310345832
PR-AUC: 0.23833327744819705


In [167]:
# ---- score new policies ----
X2 = df3[feat_cols]

#The original prototype used predict(), which returned only binary outcomes (0/1).
#To support policy ranking by risk, predict_proba() was introduced, generating a probability-based risk score instead of a binary prediction.

risk_scores = tmp.predict_proba(X2)[:, 1]

predictions = pd.DataFrame({
"policy_id": df3["policy_id"],
"risk_score": risk_scores
})

predictions.to_csv("predictions.csv", index=False)

print("done -- predictions.csv written")
print(predictions.head())
print(predictions["risk_score"].describe())

done -- predictions.csv written
    policy_id  risk_score
0  POL-015001    0.203727
1  POL-015002    0.212177
2  POL-015003    0.153395
3  POL-015004    0.149896
4  POL-015005    0.178609
count    3000.000000
mean        0.129258
std         0.074799
min         0.021318
25%         0.075646
50%         0.111627
75%         0.163091
max         0.660867
Name: risk_score, dtype: float64
